In [ ]:
import os
import urllib.request
import numpy as np
import tensorflow as tf
import random
from collections import defaultdict

# ---------------------------------------------------------------------------
# 1. Custom Pairwise NeuMF Architecture with BPR Loss & Live Accuracy Tracking
# ---------------------------------------------------------------------------

class NeuMF_BPR(tf.keras.Model):
    def __init__(self, num_users, num_items, latent_dim=16, reg_weight=1e-5):
        super(NeuMF_BPR, self).__init__()
        self.reg_weight = reg_weight

        # --- GMF Embeddings ---
        self.user_embed_gmf = tf.keras.layers.Embedding(num_users, latent_dim, name='user_gmf')
        self.item_embed_gmf = tf.keras.layers.Embedding(num_items, latent_dim, name='item_gmf')

        # --- MLP Embeddings ---
        self.user_embed_mlp = tf.keras.layers.Embedding(num_users, latent_dim * 2, name='user_mlp')
        self.item_embed_mlp = tf.keras.layers.Embedding(num_items, latent_dim * 2, name='item_mlp')

        # --- MLP Layers ---
        self.mlp_dense1 = tf.keras.layers.Dense(32, activation='relu')
        self.mlp_dense2 = tf.keras.layers.Dense(16, activation='relu')
        self.mlp_dense3 = tf.keras.layers.Dense(8, activation='relu')

        # --- Final Unified Prediction Layer ---
        self.prediction_layer = tf.keras.layers.Dense(1, use_bias=False, name='prediction')

    def predict_score(self, user_indices, item_indices):
        """Vectorized scoring method for evaluation."""
        u_gmf = self.user_embed_gmf(user_indices)
        i_gmf = self.item_embed_gmf(item_indices)
        gmf_vec = u_gmf * i_gmf

        u_mlp = self.user_embed_mlp(user_indices)
        i_mlp = self.item_embed_mlp(item_indices)
        mlp_vec = tf.keras.layers.Concatenate()([u_mlp, i_mlp])
        mlp_vec = self.mlp_dense1(mlp_vec)
        mlp_vec = self.mlp_dense2(mlp_vec)
        mlp_vec = self.mlp_dense3(mlp_vec)

        fusion_vec = tf.keras.layers.Concatenate()([gmf_vec, mlp_vec])
        return tf.squeeze(self.prediction_layer(fusion_vec), axis=-1)

    def call(self, inputs):
        user_indices, pos_item_indices, neg_item_indices = inputs
        pos_scores = self.predict_score(user_indices, pos_item_indices)
        neg_scores = self.predict_score(user_indices, neg_item_indices)
        return pos_scores, neg_scores

    def train_step(self, data):
        """Custom training step computing BPR Loss + L2 Regularization + Pairwise Accuracy."""
        inputs, _ = data
        user_indices, pos_item_indices, neg_item_indices = inputs

        with tf.GradientTape() as tape:
            pos_scores, neg_scores = self((user_indices, pos_item_indices, neg_item_indices), training=True)

            # Pairwise BPR Loss calculation
            bpr_loss = -tf.reduce_mean(tf.math.log_sigmoid(pos_scores - neg_scores))

            l2_reg = tf.add_n([
                tf.nn.l2_loss(self.user_embed_gmf.embeddings),
                tf.nn.l2_loss(self.item_embed_gmf.embeddings),
                tf.nn.l2_loss(self.user_embed_mlp.embeddings),
                tf.nn.l2_loss(self.item_embed_mlp.embeddings)
            ])
            total_loss = bpr_loss + self.reg_weight * l2_reg

            # RUNTIME ACCURACY: Percentage of pairs where pos_score > neg_score
            pairwise_accuracy = tf.reduce_mean(tf.cast(pos_scores > neg_scores, tf.float32))

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        # Returning metrics dictionary updates the live epoch print progress bar automatically
        return {"loss": total_loss, "bpr": bpr_loss, "accuracy": pairwise_accuracy}


# ---------------------------------------------------------------------------
# 2. Optimized Dataset Pipeline
# ---------------------------------------------------------------------------

def download_file(url, folder, filename):
    if not os.path.exists(folder):
        os.makedirs(folder)
    path = os.path.join(folder, filename)
    if not os.path.exists(path):
        print(f"Downloading {filename} from {url}...")
        urllib.request.urlretrieve(url, path)
    return path


def load_ngcf_format_dataset(train_file_path, test_file_path):
    train_user_dict = defaultdict(set)
    test_user_dict = defaultdict(list)
    max_user, max_item = 0, 0

    with open(train_file_path, 'r') as f:
        for line in f:
            if not line.strip(): continue
            parts = [int(x) for x in line.strip().split()]
            user = parts[0]
            items = parts[1:]
            train_user_dict[user] = set(items)
            if len(items) > 0:
                max_item = max(max_item, max(items))
            max_user = max(max_user, user)

    with open(test_file_path, 'r') as f:
        for line in f:
            if not line.strip(): continue
            parts = [int(x) for x in line.strip().split()]
            user = parts[0]
            items = parts[1:]
            test_user_dict[user] = items
            if len(items) > 0:
                max_item = max(max_item, max(items))
            max_user = max(max_user, user)

    num_users = max_user + 1
    num_items = max_item + 1
    return train_user_dict, test_user_dict, num_users, num_items


def generate_epoch_bpr_data(train_dict, num_items):
    user_indices = []
    pos_item_indices = []

    for u, items in train_dict.items():
        for i in items:
            user_indices.append(u)
            pos_item_indices.append(i)

    user_indices = np.array(user_indices, dtype=np.int32)
    pos_item_indices = np.array(pos_item_indices, dtype=np.int32)

    neg_item_indices = np.random.randint(0, num_items, size=len(user_indices), dtype=np.int32)

    for idx in range(len(user_indices)):
        u = user_indices[idx]
        pos_set = train_dict[u]
        while neg_item_indices[idx] in pos_set:
            neg_item_indices[idx] = np.random.randint(0, num_items)

    return user_indices, pos_item_indices, neg_item_indices


# ---------------------------------------------------------------------------
# 3. Optimized Metrics & Sampling Logic
# ---------------------------------------------------------------------------

def sample_negative_test_items_fast(excluded_items, num_items, num_negatives=99):
    negatives = []
    while len(negatives) < num_negatives:
        candidates = np.random.randint(0, num_items, size=num_negatives - len(negatives))
        for c in candidates:
            if c not in excluded_items and c not in negatives:
                negatives.append(c)
    return negatives


def calculate_metrics_at_10(predicted_scores):
    """
    Computes HR@10, NDCG@10, and Recall@10.
    Assumes the ground-truth target item is always located at index 0.
    """
    ranked_positions = np.argsort(predicted_scores)[::-1]
    top_10 = ranked_positions[:10]

    # 1. Hit Ratio @ 10 (Is the target item present anywhere in the top 10?)
    hit_ratio = 1.0 if 0 in top_10 else 0.0

    # 2. Recall @ 10 (Hits divided by total relevant items, which is 1 in leave-one-out setup)
    recall = 1.0 if 0 in top_10 else 0.0

    # 3. NDCG @ 10 (Normalized Discounted Cumulative Gain accounting for position position rank)
    ndcg = 0.0
    if hit_ratio == 1.0:
        rank = np.where(top_10 == 0)[0][0] + 1
        ndcg = np.log(2) / np.log(rank + 1)

    return hit_ratio, ndcg, recall


# ---------------------------------------------------------------------------
# 4. Core Experiment Pipeline
# ---------------------------------------------------------------------------

def run_experiment(dataset_name, train_path, test_path):
    print("\n" + "=" * 60)
    print(f"Running Highly-Optimized NeuMF-BPR on: {dataset_name}")
    print("=" * 60)

    train_dict, test_dict, num_users, num_items = load_ngcf_format_dataset(train_path, test_path)
    print(f"Dataset stats -> Users: {num_users}, Items: {num_items}")

    print("Pre-sampling data matrix...")
    u_idx, p_idx, n_idx = generate_epoch_bpr_data(train_dict, num_items)

    dataset = tf.data.Dataset.from_tensor_slices(((u_idx, p_idx, n_idx), np.zeros_like(u_idx)))
    dataset = dataset.shuffle(buffer_size=100000).batch(4096).prefetch(tf.data.AUTOTUNE)

    model = NeuMF_BPR(num_users, num_items, latent_dim=16, reg_weight=1e-5)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

    # Fit Model - Now displays 'loss', 'bpr', and 'accuracy' updates live!
    model.fit(dataset, epochs=5, verbose=1)

    # --- BATCHED EVALUATION ---
    print("\nStarting batched evaluation...")
    test_users = [u for u in test_dict.keys() if test_dict[u]]

    if len(test_users) > 2000:
        print("Truncating evaluation track down to 2,000 unique users for speed...")
        random.seed(42)
        test_users = random.sample(test_users, 2000)

    eval_users_list = []
    eval_items_list = []

    for u in test_users:
        target_item = test_dict[u][0]
        all_interacted = train_dict[u].union(set(test_dict[u]))
        neg_items = sample_negative_test_items_fast(all_interacted, num_items, num_negatives=99)

        eval_users_list.append(np.full(100, u, dtype=np.int32))
        eval_items_list.append(np.array([target_item] + neg_items, dtype=np.int32))

    X_users = np.concatenate(eval_users_list)
    X_items = np.concatenate(eval_items_list)

    print("Computing inference scores over total test matrix...")
    all_predictions = model.predict_score(X_users, X_items).numpy()
    all_predictions = all_predictions.reshape(-1, 100)

    total_hr = 0.0
    total_ndcg = 0.0
    total_recall = 0.0

    for preds in all_predictions:
        hr, ndcg, recall = calculate_metrics_at_10(preds)
        total_hr += hr
        total_ndcg += ndcg
        total_recall += recall

    final_hr = total_hr / len(all_predictions)
    final_ndcg = total_ndcg / len(all_predictions)
    final_recall = total_recall / len(all_predictions)

    print(f"\nFinal {dataset_name} Metrics:")
    print(f"  -> HR@10:     {final_hr:.4f}")
    print(f"  -> NDCG@10:   {final_ndcg:.4f}")
    print(f"  -> Recall@10: {final_recall:.4f}")


if __name__ == "__main__":
    gowalla_train_url = "https://raw.githubusercontent.com/xiangwang1223/neural_graph_collaborative_filtering/master/Data/gowalla/train.txt"
    gowalla_test_url = "https://raw.githubusercontent.com/xiangwang1223/neural_graph_collaborative_filtering/master/Data/gowalla/test.txt"

    gowalla_train_path = download_file(gowalla_train_url, "Data/gowalla", "train.txt")
    gowalla_test_path = download_file(gowalla_test_url, "Data/gowalla", "test.txt")

    run_experiment("Gowalla", gowalla_train_path, gowalla_test_path)

    amazon_train_url = "https://raw.githubusercontent.com/xiangwang1223/neural_graph_collaborative_filtering/master/Data/amazon-book/train.txt"
    amazon_test_url = "https://raw.githubusercontent.com/xiangwang1223/neural_graph_collaborative_filtering/master/Data/amazon-book/test.txt"

    amazon_train_path = download_file(amazon_train_url, "Data/amazon-book", "train.txt")
    amazon_test_path = download_file(amazon_test_url, "Data/amazon-book", "test.txt")

    run_experiment("Amazon-Book", amazon_train_path, amazon_test_path)



Running Highly-Optimized NeuMF-BPR on: Gowalla
Dataset stats -> Users: 29858, Items: 40981
Pre-sampling data matrix...
Epoch 1/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 16s 38ms/step - accuracy: 0.6950 - bpr: 0.5398 - loss: 0.5471
Epoch 2/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.7450 - bpr: 0.4929 - loss: 0.5036
Epoch 3/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.8280 - bpr: 0.3668 - loss: 0.3815
Epoch 4/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.9024 - bpr: 0.2314 - loss: 0.2503
Epoch 5/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.9275 - bpr: 0.1639 - loss: 0.1856

Starting batched evaluation...
Truncating evaluation track down to 2,000 unique users for speed...
Computing inference scores over total test matrix...

Final Gowalla Metrics:
  -> HR@10:     0.5675
  -> NDCG@10:   0.3477
  -> Recall@10: 0.5675

Running Highly-Optimized NeuMF-BPR on: Amazon-Book
Dataset stats -> Users: 52643, Items: 91599
Pre-sampling data matrix...
Epoch 1/